In [ ]:
import sys

!which {sys.executable}

In [ ]:
import sys
sys.path.insert(0, '..')

# Li-age relation for open clusters

https://ui.adsabs.harvard.edu/abs/2024A%26A...685A..83G/abstract

https://cdsarc.cds.unistra.fr/viz-bin/cat/J/A+A/685/A83

In [ ]:
from wakai.catalog import VIZIER_KEYS_LiEW_CATALOG

#add custom catalog for Lithium
VIZIER_KEYS_LiEW_CATALOG['GutierrezAlbarran2024'] = "J/A+A/685/A83"

In [ ]:
from wakai.catalog import CatalogDownloader

cd = CatalogDownloader(catalog_name="GutierrezAlbarran2024", catalog_type="liew")
tabs = cd.get_tables_from_vizier(row_limit=-1)

In [ ]:
df1 = tabs[0].to_pandas()
df1.head()

In [ ]:
df = tabs[1].to_pandas()
df.head()

In [ ]:
df.shape

In [ ]:
df.columns.tolist()

In [ ]:
df = df1[['Cluster','Name']].merge(df, left_on='Cluster', right_on='No', how='left')
df.head()

In [ ]:
import matplotlib.pyplot as pl
import numpy as np

fig, ax = pl.subplots()

groups = df.groupby(by='Name')
cmap = pl.get_cmap('hsv')
colors = [cmap(i) for i in np.linspace(0, 1, len(groups))]

for i,(name, group) in enumerate(groups):
    group.plot.scatter(x='Teff', y='EWcLi', color=colors[i], ax=ax, label=name)
pl.legend()

In [ ]:
ax = df.plot.scatter(x='__Fe_H_', y='EWcLi')

# LAMOST

https://ui.adsabs.harvard.edu/abs/2022A%26A...668A..49F/abstract

In [ ]:
from wakai.catalog import VIZIER_KEYS_LiEW_CATALOG

#add custom catalog for Lithium
VIZIER_KEYS_LiEW_CATALOG['Franciosini2022'] = "J/A+A/668/A49"

In [ ]:
from wakai.catalog import CatalogDownloader

cd = CatalogDownloader(catalog_name="Franciosini2022", catalog_type="liew")
tabs = cd.get_tables_from_vizier(row_limit=-1)

In [ ]:
import pandas as pd

df1 = tabs[0].to_pandas()
df2 = tabs[2].to_pandas()
df = pd.concat([df1,df2])

In [ ]:
ax = df.plot.scatter(x='Teff', y='EW_Li_')

In [ ]:
ax = df.plot.scatter(x='__Fe_H_', y='A_Li_')

TODO: table for Li EW for published planets

In [ ]:
import numpy as np

# Constants
R_H = 1.0973731568508e7  # Rydberg constant for hydrogen in m^-1
c = 2.998e8              # Speed of light in m/s
h = 6.626e-34            # Planck's constant in J*s
eV = 1.602e-19           # Electron volt in Joules

# Function to calculate the wavelength of the emission/absorption lines using Rydberg formula
def calculate_wavelength(n_initial, n_final):
    """
    Calculate the wavelength for transitions between two energy levels in hydrogen-like atoms.

    Parameters:
    n_initial (int): Principal quantum number of the initial state (higher energy level)
    n_final (int): Principal quantum number of the final state (lower energy level)

    Returns:
    float: Wavelength in meters
    """
    # Calculate the wavelength using the Rydberg formula
    wavelength = 1 / (R_H * (1/n_final**2 - 1/n_initial**2))
    return wavelength

# Function to print wavelengths for the Balmer series (visible spectrum)
def balmer_series(n_final=1):
    """
    Print the wavelengths of the Balmer series for hydrogen, corresponding to transitions
    from n = 3, 4, 5, 6... down to n = 2.
    """
    print("Balmer Series (n_final = 2) Wavelengths (in nm):")
    for n_initial in range(3, 7):  # Calculating for n_initial = 3, 4, 5, 6
        wavelength_m = calculate_wavelength(n_initial, n_final)
        wavelength_nm = wavelength_m * 1e9  # Convert meters to nanometers
        print(f"Transition from n={n_initial} to n={n_final}: {wavelength_nm:.2f} nm")

balmer_series()

# Li EW

In [ ]:
# %load init_ipynb
import os
import sys
import matplotlib.pyplot as pl

sys.path.insert(0, '..')
assert os.environ["CONDA_DEFAULT_ENV"] == 'wakai'

plotpar = {'axes.labelsize': 16,
           'font.size': 16,
           'legend.fontsize': 16,
           'xtick.labelsize': 16,
           'ytick.labelsize': 16,
           #'text.usetex': False,
           'xtick.direction': 'in',
           'ytick.direction': 'in'
           }
pl.rcParams.update(plotpar)


## Stanford-Moore+2020 (field)

In [ ]:
from wakai.catalog import CatalogDownloader

cd = CatalogDownloader(catalog_name="Stanford-Moore2020", catalog_type="LiEW")
tabs = cd.get_tables_from_vizier(row_limit=-1)

In [ ]:
df = tabs[0].to_pandas()
df

## Grandjean+2020, 21, 23 (field)
* merge targets and query Li and/or logRHK from vizier because they are not included in the published table

In [ ]:
from astroquery.vizier import Vizier
import pandas as pd

tables = {}
for i, key in enumerate(["J/A+A/669/A12", "J/A+A/650/A39", "J/A+A/633/A44"]):
    tabs = Vizier.get_catalogs(key)
    tables[i] = tabs[0].to_pandas()

In [ ]:
df = pd.concat(tables).reset_index(drop=True)
df.tail()

In [ ]:
from astropy.coordinates import Angle
from tqdm import tqdm
from wakai.catalog import Target
import numpy as np

params = ["B-V", "log_R_HK_", "EWLi"]

data = pd.DataFrame()
for i,row in tqdm(df.iterrows()):

    ra = Angle(row.RAJ2000 , unit='hourangle')
    dec = Angle(row.DEJ2000 , unit='degree')
    t = Target(ra.deg, dec.deg)
    
    d = {}
    d['Name'] = row.Name
    for param in params:
        p = t.query_vizier_param(param)
        v = np.nanmean(list(p.values()))
        d[param] = v
    data = data.append(d, ignore_index=True)

In [ ]:
data.dropna()

## Gutierrez+2020 (cluster)

In [ ]:
from wakai.catalog import CatalogDownloader

cd = CatalogDownloader(catalog_name="Gutierrez2020", catalog_type="LiEW")
tabs = cd.get_tables_from_vizier(row_limit=-1)

See Fig 4.7 from [Guttierez's thesis](https://ui.adsabs.harvard.edu/abs/2022PhDT.........6G/abstract)
![img](../figures/LiTeff_GuttierezLuthien2022.png)

In [ ]:
import pandas as pd
from pathlib import Path

fp = Path('../data/Gutierrez2020_orig.csv')
if not fp.exists():
    url = "https://www.aanda.org/articles/aa/full_html/2020/11/aa37620-20/T1.html"
    html = pd.read_html(url, #encoding ='us-ascii'
                       )
    df = html[0].dropna()
    df['Age'] = df.Age.replace(' \\(b\\)','', regex=True)
    df['Name'] = """rhoOph cha gammaVel ngc2457 ic2391 ic2602 ic4665 ngc2516 ngc6705 ngc4815 
                    ngc6633 tr23 br81 ngc6005 ngc6802 pismis18 tr20 br44 m67 ngc2243""".split()
    df.to_csv(fp)
else:
    df = pd.read_csv(fp, delimiter='\t')
    df['Name'] = """0 rhoOph cha gammaVel ngc2457 ic2391 ic2602 ic4665 ngc2516 ngc6705 ngc4815 
                    ngc6633 tr23 br81 ngc6005 ngc6802 pismis18 tr20 br44 m67 ngc2243""".split()
df.head()

In [ ]:
fig, axs = pl.subplots(2,2, figsize=(8,8), tight_layout=True, sharex='row', sharey='col')

dfs = {}
for i,key in enumerate(df.Name.values):
    ax = axs.flatten()[i//5]
    
    name = key.split('/')[-1]
    age = df.iloc[i]['Age'].values[0]
    d = tabs[i].to_pandas()
    #remove non-members
    try:
        d = d[d.Final=='Y']
    except Exception as e:
        print(name,e)
    _ = d.plot(x='Teff', y='EW_Li_', kind='scatter', ax=ax, c=f"C{i}", label=f"{name} ({age})")
    ax.legend(fontsize=12)
    ax.invert_xaxis()
    dfs[name] = d

In [ ]:
cols = ['Name', 'CNAME', 'RV', 'e_RV', 'Teff', 'e_Teff', 'logg', 'e_logg', 
        '__Fe_H_', 'e__Fe_H_', 
        #'l_EW_Li_', 
        'EW_Li_', 'e_EW_Li_', 
        #'f_e_EW_Li_'
       ]
left = pd.concat(dfs).reset_index().drop(['level_1', 'ID'], axis=1)
left = left.rename({'level_0': 'Name'}, axis=1)[cols]

right = pd.Series({'rhoOph': '1–3',
                      'Cha': '2',
                      'gammaVel': '10–20',
                      'NGC 2547':  '35–45',
                      'IC 2391': '36 ± 2',
                      'IC 2602': '35 ± 1',
                      'IC 4665': '38 ± 3',
                      'NGC 2516': '251 ± 3',
                      'NGC 6705': '300 ± 50',
                      'NGC 4815': '570 ± 70',
                      'NGC 6633': '773 ± 10',
                      'Trumpler 23': '800 ± 100',
                      'Berkeley 81' : '860 ± 100',
                      'NGC 6005': '973 ± 4',
                      'NGC 6802': '1000 ± 100',
                      'Pismis 18': '1200 ± 400',
                      'Trumpler 20': '1500 ± 150',
                      'Berkeley 44': '1600 ± 300',
                      'M67': '4000-4500',
                      'NGC 2243': '4000 ± 120'}, name='Age'
                   )
right

In [ ]:
df_final = pd.merge(left, right, on='Name')
# df_final = df_final.apply(pd.to_numeric)
# df_final = df_final.applymap(lambda x: x.decode("ascii") if isinstance(x, bytes) else str(x))
# df_final.Age = df_final.Age.str.replace('\W', '-', regex=True)
# df_final.Distance = df_final.Distance.str.replace('\W', '.', regex=True)
df_final.tail()

In [ ]:
df_final.to_csv("../data/Li_catalog_Gutierrez2020.csv", encoding="us-ascii")

# Li abundances

## Franciosini+2022

In [ ]:
from wakai.catalog import CatalogDownloader

cd = CatalogDownloader(catalog_name="Franciosini2022", catalog_type="LiEW")
tabs = cd.get_tables_from_vizier(row_limit=-1)

In [ ]:
cnames = "25Ori GammaVelA GammaVelB NGC2451B NGC2547 NGC2516".split()
dfs = {}
for i,tab in enumerate(tabs):
    name = cnames[i]    
    df = tab.to_pandas()
    
    if i==0:
        ax=None
    ax = df.plot(x='Teff', y='ALi', kind='scatter', ax=ax, c=f"C{i}", label=name)
    ax.legend()
    ax.invert_xaxis()
    
    dfs[name] = df

merge A Li from Deliyannis+2019

In [ ]:
cd = CatalogDownloader(catalog_name="Deliyannis2019", catalog_type="LiEW")
tabs = cd.get_tables_from_vizier(row_limit=-1)

In [ ]:
d = tabs[0].to_pandas()
d.head()

In [ ]:
dfs['25Ori'].columns

In [ ]:
d.columns

In [ ]:
d = d.rename({'l_A_Li_': 'l_ALi', 
          'A_Li_': 'ALi',
          'e_A_Li_': 'e_ALi'
         }, axis=1)

In [ ]:
dfs['NGC6819'] = d

In [ ]:
pd.concat(dfs)

## Magrini+2021

In [ ]:
from wakai.catalog import CatalogDownloader
import numpy as np

cd = CatalogDownloader(catalog_name="Magrini2021", catalog_type="LiEW")
tabs = cd.get_tables_from_vizier(row_limit=-1)

In [ ]:
df = tabs[0].to_pandas()
df.head()

In [ ]:
df.plot(x='Teff', y='Li1', kind='scatter')

In [ ]:
from utils import get_nexsci_data

df_nexsci = get_nexsci_data(table_name="ps", clobber=False)
# df_nexsci.columns.tolist()

In [ ]:
from wakai.catalog import CatalogDownloader
import numpy as np

cd = CatalogDownloader(catalog_name="Bouvier2018_Pleiades", catalog_type="LiEW")
tabs = cd.get_tables_from_vizier(row_limit=-1)
df_Pleiades = tabs[0].to_pandas()
df_Pleiades['logTeff'] = df_Pleiades.Teff.apply(lambda x: np.log10(x))
df_Pleiades['EW_Li_'] = df_Pleiades['WLi'].copy()
# df_PisEri.head()
df_Pleiades

In [ ]:
cd = CatalogDownloader(catalog_name="Cummings2017_HyadesPraesepe", catalog_type="LiEW")
tabs = cd.get_tables_from_vizier(row_limit=-1)
df_Hyades = tabs[0].to_pandas()
df_Praesepe = tabs[1].to_pandas()
df_Hyades['logTeff'] = df_Hyades.Teff.apply(lambda x: np.log10(x))
df_Praesepe['logTeff'] = df_Praesepe.Teff.apply(lambda x: np.log10(x))
# df_PisEri['st_rotp'] = df_PisEri['Prot'].copy()
# df_PisEri.head()
df_Hyades

In [ ]:
df_Praesepe

In [ ]:
from utils import plot_planets

x, y ='logTeff', 'EW_Li_'
ax = plot_planets(df_Pleiades, x, y, 
                plot_kwds={'marker': '.', 'c':'C1', 'label': 'Pleiades (0.12 Gyr)'}
                                            )
                                            
_ =  plot_planets(df_Praesepe, x, y, ax=ax,
                plot_kwds={'marker': '.', 'c':'C2', 'label': 'Praepe (0.67 Gyr)'}
                                            )

_ =  plot_planets(df_Hyades, x, y, ax=ax,
                plot_kwds={'marker': '.', 'c':'C3', 'label': 'Hyades (0.73 Gyr)'}
                                             )

In [ ]:
fp = "../figures/EW_Teff.png"
ax.figure.savefig(fp, bbox_inches="tight")